In [2]:
"""
01 - Explore ParlaMint-NL-en data structure

Goal: understand the XML structure of a single file before scaling up.
"""

import os
from pathlib import Path
from lxml import etree

# Robust path resolution: locate project root regardless of where Jupyter starts
PROJECT_ROOT = Path.cwd()
# If we're inside notebooks/, go up one level
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")

# Paths
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ParlaMint-NL-en.TEI.ana"
SAMPLE_FILE = DATA_DIR / "2022" / "ParlaMint-NL-en_2022-01-18-tweedekamer-6.ana.xml"

print(f"\nData dir exists: {DATA_DIR.exists()}")
print(f"Sample file exists: {SAMPLE_FILE.exists()}")

if SAMPLE_FILE.exists():
    print(f"Sample file size: {SAMPLE_FILE.stat().st_size / (1024*1024):.2f} MB")
else:
    # Diagnostic: list what's in the data dir
    print("\nFiles in 2022 folder (first 5):")
    if (DATA_DIR / "2022").exists():
        for f in sorted((DATA_DIR / "2022").iterdir())[:5]:
            print(f"  {f.name}")

Project root: c:\Users\becla\Documents\projects\hcss-parlamint-climate-security
Working directory: c:\Users\becla\Documents\projects\hcss-parlamint-climate-security

Data dir exists: True
Sample file exists: True
Sample file size: 2.52 MB


In [3]:
"""
Explore the XML structure of one file.

ParlaMint files use TEI XML. I need to understand:
- Where the speeches are 
- How speakers are identified
- Where metadata lives (date, party affiliation)
- What linguistic annotations are present 
"""

# Parse the file
tree = etree.parse(str(SAMPLE_FILE))
root = tree.getroot()

print(f"Root element: {root.tag}")
print(f"Root attributes: {dict(root.attrib)}")
print(f"\nNamespaces: {root.nsmap}")

# Print high-level structure 
print("\nDirect children of root:")
for child in root:
    print(f"  - {child.tag}")

Root element: {http://www.tei-c.org/ns/1.0}TEI
Root attributes: {'{http://www.w3.org/XML/1998/namespace}id': 'ParlaMint-NL-en_2022-01-18-tweedekamer-6.ana', '{http://www.w3.org/XML/1998/namespace}lang': 'en', 'ana': '#parla.sitting #covid', 'corresp': '../../ParlaMint-NL.TEI.ana/2022/ParlaMint-NL_2022-01-18-tweedekamer-6.ana.xml'}

Namespaces: {None: 'http://www.tei-c.org/ns/1.0'}

Direct children of root:
  - {http://www.tei-c.org/ns/1.0}teiHeader
  - {http://www.tei-c.org/ns/1.0}text


In [4]:
"""
Explore the two main branches: teiHeader and text.

I define a namespace map (NS) so I can write XPath queries cleanly.
The 'tei' prefix below is just a convenience.
"""

# Define namespace map for clean XPath queries
NS = {"tei": "http://www.tei-c.org/ns/1.0"}

# Get teiHeader and text
header = root.find("tei:teiHeader", NS)
text = root.find("tei:text", NS)

print("=" * 60)
print("TEI HEADER STRUCTURE (first 2 levels)")
print("=" * 60)

def print_tree(elem, depth=0, max_depth=2):
    """Print element tree up to max_depth levels."""
    tag = elem.tag.split("}")[-1]  # strip namespace for readability
    attrs = " ".join(f'{k.split("}")[-1]}="{v[:30]}"' for k, v in elem.attrib.items())
    print("  " * depth + f"<{tag} {attrs}>")
    if depth < max_depth:
        for child in elem:
            print_tree(child, depth + 1, max_depth)

print_tree(header, max_depth=2)

print("\n" + "=" * 60)
print("TEXT STRUCTURE (first 2 levels)")
print("=" * 60)
print_tree(text, max_depth=2)

TEI HEADER STRUCTURE (first 2 levels)
<teiHeader >
  <fileDesc >
    <titleStmt >
    <editionStmt >
    <extent >
    <publicationStmt >
    <sourceDesc >
  <encodingDesc >
    <projectDesc >
    <tagsDecl >
    <listPrefixDef >
  <profileDesc >
    <settingDesc >

TEXT STRUCTURE (first 2 levels)
<text lang="nl" ana="#covid">
  <body >
    <div type="debateSection" lang="en">


In [5]:
"""
Drill into the debate section: find utterances <u> and inspect one.

In TEI/ParlaMint:
- <u> = utterance (a single speech turn by one speaker)
- <seg> = segment (a unit within an utterance, typically a sentence or paragraph)
- @who = attribute pointing to the speaker
- @ana = attribute with annotations 
"""

# Find all <u> (utterances) in the file
utterances = text.findall(".//tei:u", NS)
print(f"Total utterances in this file: {len(utterances)}")

# Look at the first utterance in detail
first_u = utterances[0]
print(f"\nFirst utterance attributes: {dict(first_u.attrib)}")

# Get the text content of the first utterance
# In .ana files, the text is inside <seg> elements (segments)
segments = first_u.findall("tei:seg", NS)
print(f"Number of segments in first utterance: {len(segments)}")

# Show the first segment's text (just the plain text, not annotations)
if segments:
    first_seg = segments[0]
    # itertext() concatenates all text content, ignoring annotation tags
    seg_text = "".join(first_seg.itertext()).strip()
    print(f"\nFirst segment text (first 500 chars):")
    print(seg_text[:500])

Total utterances in this file: 171

First utterance attributes: {'ana': '#chair', '{http://www.w3.org/XML/1998/namespace}id': 'ParlaMint-NL_2022-01-18-tweedekamer-6.u1', 'who': '#VeraBergkamp', '{http://www.w3.org/XML/1998/namespace}lang': 'en', 'corresp': 'mt-src:ParlaMint-NL_2022-01-18-tweedekamer-6.u1'}
Number of segments in first utterance: 1

First segment text (first 500 chars):
The
                     
next
item
                     
is
the
debate
on
the
declaration
of
government
.
                  
                  
I
invite
                     
Mr
Wilders
                     
from
the
                     
PVV
                     
to
                     
give
his
                     
input
.


In [6]:
"""
Extract metadata from the teiHeader: 
- Title of the meeting
- Date
- Subcorpus tag (reference vs covid)
- List of speakers present in this meeting with their party
"""

# Title of the meeting
title = header.find(".//tei:titleStmt/tei:title[@type='main']", NS)
print(f"Title: {title.text if title is not None else 'N/A'}")

# Date of the meeting
date_elem = header.find(".//tei:settingDesc//tei:date", NS)
print(f"Date attrs: {dict(date_elem.attrib) if date_elem is not None else 'N/A'}")
print(f"Date text: {date_elem.text if date_elem is not None else 'N/A'}")

# Word count
extent = header.find(".//tei:extent", NS)
if extent is not None:
    measures = extent.findall("tei:measure", NS)
    for m in measures:
        print(f"Extent measure: {m.attrib.get('unit')} = {m.text}")

# All <ana> values on the root tell us subcorpus + meeting type
print(f"\nRoot ana attribute: {root.attrib.get('ana')}")

# Look at first 3 utterances to see different speaker roles (chair, regular, etc.)
print("\nFirst 5 utterances (speaker, role):")
for u in utterances[:5]:
    speaker = u.attrib.get("who", "?")
    role = u.attrib.get("ana", "")
    u_id = u.attrib.get("{http://www.w3.org/XML/1998/namespace}id", "?")
    print(f"  {u_id}: speaker={speaker}, role={role}")

Title: Dutch parliamentary corpus ParlaMint-NL-en, Lower House 2022-01-18 [ParlaMint-en.ana]
Date attrs: {'when': '2022-01-18'}
Date text: 2022-01-18
Extent measure: speeches = 171 speeches
Extent measure: speeches = 171 spreekbeurten
Extent measure: words = 12,661 words
Extent measure: words = 12,661 woorden

Root ana attribute: #parla.sitting #covid

First 5 utterances (speaker, role):
  ParlaMint-NL_2022-01-18-tweedekamer-6.u1: speaker=#VeraBergkamp, role=#chair
  ParlaMint-NL_2022-01-18-tweedekamer-6.u2: speaker=#GeertWilders, role=#regular
  ParlaMint-NL_2022-01-18-tweedekamer-6.u3: speaker=#VeraBergkamp, role=#chair
  ParlaMint-NL_2022-01-18-tweedekamer-6.u4: speaker=#JanPaternotte, role=#regular
  ParlaMint-NL_2022-01-18-tweedekamer-6.u5: speaker=#GeertWilders, role=#regular


In [7]:
"""
Explore ParlaMint-NL-listPerson.xml — the file that maps speaker IDs to their full info.

The meeting files only have speaker IDs like "#VeraBergkamp",
but to filter by party / political orientation we need to look up each speaker here.
"""

LIST_PERSON_FILE = DATA_DIR / "ParlaMint-NL-listPerson.xml"
print(f"File exists: {LIST_PERSON_FILE.exists()}")
print(f"File size: {LIST_PERSON_FILE.stat().st_size / 1024:.1f} KB")

# Parse it
person_tree = etree.parse(str(LIST_PERSON_FILE))
person_root = person_tree.getroot()

# Find all <person> elements
persons = person_root.findall(".//tei:person", NS)
print(f"\nTotal persons: {len(persons)}")

# Inspect the first person to understand the structure
print("\n--- Structure of one example person (Geert Wilders) ---")
# Find Wilders specifically
wilders = None
for p in persons:
    pid = p.attrib.get("{http://www.w3.org/XML/1998/namespace}id", "")
    if "Wilders" in pid:
        wilders = p
        break

if wilders is not None:
    # Print the full XML of this person element (formatted)
    print(etree.tostring(wilders, pretty_print=True, encoding="unicode")[:3000])
else:
    # Fallback: show the first person
    print(etree.tostring(persons[0], pretty_print=True, encoding="unicode")[:3000])

File exists: True
File size: 327.7 KB

Total persons: 586

--- Structure of one example person (Geert Wilders) ---
<person xmlns="http://www.tei-c.org/ns/1.0" n="127" xml:id="GeertWilders">
      <persName>
         <surname>Wilders</surname>
         <forename>Geert</forename>
      </persName>
      <sex value="M"/>
      <affiliation role="member" ref="#party.PVV" from="2006-02-22"/>
      <birth when="1963-09-06">
         <placeName ref="https://www.geonames.org/2745641">Venlo</placeName>
      </birth>
      <idno type="URI" subtype="wikimedia">https://nl.wikipedia.org/wiki/Geert_Wilders</idno>
   </person>
   



In [8]:
"""
Explore ParlaMint-NL-listOrg.xml — the file with all organizations (parties + ministries).

I need this to:
1. Map party IDs like "#party.PVV" to full party names
2. Get political orientation if available
"""

LIST_ORG_FILE = DATA_DIR / "ParlaMint-NL-listOrg.xml"
print(f"File size: {LIST_ORG_FILE.stat().st_size / 1024:.1f} KB")

org_tree = etree.parse(str(LIST_ORG_FILE))
org_root = org_tree.getroot()

# Find all <org> elements
orgs = org_root.findall(".//tei:org", NS)
print(f"Total organizations: {len(orgs)}")

# Filter to parties only (role="politicalParty")
parties = [o for o in orgs if o.attrib.get("role") == "politicalParty"]
print(f"Total political parties: {len(parties)}")

# Show one party in detail
print("\n--- Structure of one example party ---")
if parties:
    pvv = next((p for p in parties if "PVV" in p.attrib.get("{http://www.w3.org/XML/1998/namespace}id", "")), parties[0])
    print(etree.tostring(pvv, pretty_print=True, encoding="unicode")[:2000])

# Print all party IDs and abbreviated names
print("\n--- All Dutch parties in the corpus ---")
for p in parties:
    pid = p.attrib.get("{http://www.w3.org/XML/1998/namespace}id", "?")
    # Try to find the abbreviated form
    abbr = p.find("tei:orgName[@full='abb']", NS)
    full_name = p.find("tei:orgName[@full='yes']", NS)
    abbr_text = abbr.text if abbr is not None else "?"
    full_text = full_name.text if full_name is not None else "?"
    print(f"  {pid} | {abbr_text} | {full_text}")

File size: 183.0 KB
Total organizations: 50
Total political parties: 0

--- Structure of one example party ---

--- All Dutch parties in the corpus ---


In [9]:
"""
Diagnostic: see what types of organizations exist in listOrg.xml,
since 'politicalParty' returned 0 results.
"""

# Get all unique 'role' values from the orgs
from collections import Counter

roles = Counter()
for o in orgs:
    role = o.attrib.get("role", "(no role)")
    roles[role] += 1

print("Distinct 'role' values among organizations:")
for role, count in roles.most_common():
    print(f"  {role}: {count}")

# Show one full org element to see the actual structure
print("\n--- Structure of first org element ---")
print(etree.tostring(orgs[0], pretty_print=True, encoding="unicode")[:2000])

# Show first 3 with their roles + IDs
print("\n--- First 5 orgs (id, role, name) ---")
for o in orgs[:5]:
    oid = o.attrib.get("{http://www.w3.org/XML/1998/namespace}id", "?")
    role = o.attrib.get("role", "?")
    name_elem = o.find("tei:orgName", NS)
    name = name_elem.text if name_elem is not None else "?"
    print(f"  {oid} | role={role} | name={name}")

Distinct 'role' values among organizations:
  parliamentaryGroup: 35
  ministry: 12
  parliament: 2
  government: 1

--- Structure of first org element ---
<org xmlns="http://www.tei-c.org/ns/1.0" xml:id="GOV" role="government">
      <orgName xml:lang="nl" full="yes">Regering van Nederland</orgName>
      <orgName xml:lang="en" full="yes">Government of the Netherlands</orgName>
      <event from="1815-09-21">
         <label xml:lang="en">existence</label>
      </event>
      <listEvent>
         <event xml:id="GOV.28" from="2012-11-05" to="2017-03-14">
            <label xml:lang="en">Term 28</label>
            <label xml:lang="nl">Nummer 28</label>
         </event>
         <event xml:id="GOV.29" from="2017-10-26" to="2022-01-09">
            <label xml:lang="en">Term 29</label>
            <label xml:lang="nl">Nummer 29</label>
         </event>
         <event xml:id="GOV.30" from="2022-01-10">
            <label xml:lang="en">Term 30</label>
            <label xml:lang="nl">Nu

In [10]:
"""
Now correctly extract all parliamentary groups (= political parties).
Each group has multiple <orgName> elements: 'abb' (abbreviation), 'init' (initials),
and 'yes' (full name). I want abbreviation + full English name.
"""

# Filter to parliamentary groups (= parties)
parties = [o for o in orgs if o.attrib.get("role") == "parliamentaryGroup"]
print(f"Total parties: {len(parties)}\n")

# Build a clean party dictionary: id -> {abbr, full_name_en, full_name_nl}
party_info = {}
for p in parties:
    pid = p.attrib.get("{http://www.w3.org/XML/1998/namespace}id", "?")
    info = {"id": pid, "abbr": None, "name_en": None, "name_nl": None}
    
    for name_elem in p.findall("tei:orgName", NS):
        full_attr = name_elem.attrib.get("full")
        lang = name_elem.attrib.get("{http://www.w3.org/XML/1998/namespace}lang")
        text = (name_elem.text or "").strip()
        
        if full_attr == "abb":
            info["abbr"] = text
        elif full_attr == "yes" and lang == "en":
            info["name_en"] = text
        elif full_attr == "yes" and lang == "nl":
            info["name_nl"] = text
    
    party_info[pid] = info

# Print all parties nicely
print(f"{'ID':<25} {'Abbr':<10} {'Full name (EN)':<60}")
print("-" * 100)
for pid, info in sorted(party_info.items()):
    print(f"{info['id']:<25} {info['abbr'] or '':<10} {info['name_en'] or '':<60}")

Total parties: 35

ID                        Abbr       Full name (EN)                                              
----------------------------------------------------------------------------------------------------
party.50PLUS              50PLUS                                                                 
party.BBB                 BBB                                                                    
party.BIJ1                BIJ1                                                                   
party.Bontes              Bontes                                                                 
party.CDA                 CDA                                                                    
party.ChristenUnie        CU                                                                     
party.D66                 D66                                                                    
party.DENK                DENK                                                                  

In [11]:
"""
Test the parser on a small subset (50 files) to verify everything works.
If this looks good, run on all 6100 files.
"""

import sys
sys.path.insert(0, str(PROJECT_ROOT))

# Force reimport in case we edit the module
if "src.data_loader" in sys.modules:
    del sys.modules["src.data_loader"]

from src.data_loader import build_corpus_dataframe

CORPUS_ROOT = PROJECT_ROOT / "data" / "raw" / "ParlaMint-NL-en.TEI.ana"
LIST_PERSON = CORPUS_ROOT / "ParlaMint-NL-listPerson.xml"
LIST_ORG = CORPUS_ROOT / "ParlaMint-NL-listOrg.xml"

# Quick test: parse only 50 files
df_sample = build_corpus_dataframe(
    corpus_root=CORPUS_ROOT,
    list_person_path=LIST_PERSON,
    list_org_path=LIST_ORG,
    limit=50,
)

print(f"\nDataFrame shape: {df_sample.shape}")
print(f"Columns: {list(df_sample.columns)}")
df_sample.head()

Loading speaker metadata...
  586 speakers
Loading party metadata...
  35 parties

Found 50 meeting files. Parsing...


100%|██████████| 50/50 [00:07<00:00,  6.46it/s]



Total speeches: 3,477
Resolving speaker → party at meeting date...

DataFrame shape: (3477, 14)
Columns: ['speech_id', 'file_id', 'date', 'year', 'house', 'subcorpus', 'speaker_id', 'role', 'title', 'text', 'n_words', 'speaker_name', 'party_id', 'party_abbr']


,speech_id,file_id,date,year,house,subcorpus,speaker_id,role,title,text,n_words,speaker_name,party_id,party_abbr
0,ParlaMint-NL_2014-04-16-tweedekamer-2.u1,ParlaMint-NL-en_2014-04-16-tweedekamer-2.ana,2014-04-16,2014,Lower,reference,AnouchkavanMiltenburg,chair,"Dutch parliamentary corpus ParlaMint-NL-en, Lo...",There are now enough members who have also sig...,76,Anouchka van Miltenburg,party.VVD,VVD
1,ParlaMint-NL_2014-04-16-tweedekamer-2.u2,ParlaMint-NL-en_2014-04-16-tweedekamer-2.ana,2014-04-16,2014,Lower,reference,FleurAgema,regular,"Dutch parliamentary corpus ParlaMint-NL-en, Lo...",Mr. Chairman . I 'll file the next motion .,10,Fleur Agema,party.PVV,PVV
2,ParlaMint-NL_2014-04-16-tweedekamer-2.u3,ParlaMint-NL-en_2014-04-16-tweedekamer-2.ana,2014-04-16,2014,Lower,reference,MartineBaay-Timmerman,regular,"Dutch parliamentary corpus ParlaMint-NL-en, Lo...",Mr. Chairman . I 'll file the following motions .,10,Martine Baay-Timmerman,party.50PLUS,50PLUS
3,ParlaMint-NL_2014-04-16-tweedekamer-2.u4,ParlaMint-NL-en_2014-04-16-tweedekamer-2.ana,2014-04-16,2014,Lower,reference,MartinvanRijn,regular,"Dutch parliamentary corpus ParlaMint-NL-en, Lo...",Mr. Chairman . I will respond to the motions t...,179,Martin van Rijn,party.PvdA,PvdA
4,ParlaMint-NL_2014-04-16-tweedekamer-2.u5,ParlaMint-NL-en_2014-04-16-tweedekamer-2.ana,2014-04-16,2014,Lower,reference,FleurAgema,regular,"Dutch parliamentary corpus ParlaMint-NL-en, Lo...",So that was exactly not my point . Of course i...,108,Fleur Agema,party.PVV,PVV


In [12]:
"""
Run the parser on the FULL corpus (all 6100 files) and save to CSV.
This takes ~15 minutes. 
"""

# Force reimport in case data_loader.py was edited
if "src.data_loader" in sys.modules:
    del sys.modules["src.data_loader"]
from src.data_loader import build_corpus_dataframe

# Parse the FULL corpus (no limit)
df_full = build_corpus_dataframe(
    corpus_root=CORPUS_ROOT,
    list_person_path=LIST_PERSON,
    list_org_path=LIST_ORG,
    limit=None,  # all files
)

# Save to processed/ folder as CSV
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "all_speeches.csv"
df_full.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"\nSaved {len(df_full):,} speeches to:")
print(f"  {OUTPUT_PATH}")
print(f"  File size: {OUTPUT_PATH.stat().st_size / (1024*1024):.1f} MB")

Loading speaker metadata...
  586 speakers
Loading party metadata...
  35 parties

Found 6100 meeting files. Parsing...


100%|██████████| 6100/6100 [20:50<00:00,  4.88it/s]   



Total speeches: 609,209
Resolving speaker → party at meeting date...

Saved 609,209 speeches to:
  c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\data\processed\all_speeches.csv
  File size: 517.1 MB


In [2]:
"""
Sanity check: load the saved CSV and look at distributions.
"""
from pathlib import Path
import pandas as pd

# Resolve project root robustly (works regardless of where Jupyter starts)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED_DIR / "all_speeches.csv")
print(f"Total speeches: {len(df):,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nSpeeches per year:")
print(df["year"].value_counts().sort_index())
print(f"\nSpeeches per house:")
print(df["house"].value_counts())
print(f"\nSpeeches per subcorpus:")
print(df["subcorpus"].value_counts())
print(f"\nTop 10 parties by speech count:")
print(df["party_abbr"].value_counts().head(10))
print(f"\nMissing party_abbr (no party affiliation found):")
print(f"  {df['party_abbr'].isna().sum():,} speeches ({df['party_abbr'].isna().mean()*100:.1f}%)")

Total speeches: 609,209
Date range: 2014-04-16 to 2022-07-12

Speeches per year:
year
2014     4710
2015    73100
2016    78156
2017    63078
2018    97083
2019    97230
2020    76736
2021    74062
2022    45054
Name: count, dtype: int64

Speeches per house:
house
Lower    532241
Upper     76968
Name: count, dtype: int64

Speeches per subcorpus:
subcorpus
reference    421061
covid        188148
Name: count, dtype: int64

Top 10 parties by speech count:
party_abbr
PvdA    151532
VVD     122661
D66      82874
CDA      57564
SP       43188
GL       32551
PVV      32041
CU       24125
PvdD     12948
SGP      12204
Name: count, dtype: int64

Missing party_abbr (no party affiliation found):
  850 speeches (0.1%)
